In [1]:
import os
import sys
import pandas as pd
import numpy as np
import yaml
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from modeling import run_base_learner_suite
from preprocessing import split_column_types

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
PROCESSED_DIR = os.path.join(PROJECT_ROOT, config["paths"]["processed_dir"])
MODELS_ROOT = os.path.join(PROJECT_ROOT, config["paths"]["models_dir"])
TABLES_DIR = os.path.join(PROJECT_ROOT, config["paths"]["tables_dir"])

MODEL_NAMES = ["dt", "rf", "gb", "svm", "nb", "knn", "xgb"]
N_TRIALS_TREE = config["optuna"]["n_trials_tree"]
N_TRIALS_SVM = config["optuna"]["n_trials_svm"]
TRIAL_BUDGET = {"dt": N_TRIALS_TREE, "rf": N_TRIALS_TREE, "gb": N_TRIALS_TREE, "xgb": N_TRIALS_TREE,
                 "svm": N_TRIALS_SVM, "knn": N_TRIALS_SVM, "nb": N_TRIALS_SVM}
SVM_FIT_ROWS = config["svm_subsample"]["fit_rows"]
N_BOOTSTRAP = config["bootstrap"]["n_resamples"]

NUMERIC_COLS = ["WomenAge", "MariageAge", "NumberofPregnancy", "NumberofAbortions",
                "CovidDiagL1", "CovidDiagL2", "SleepingAfter", "BMI_Pregnancy"]
CATEGORICAL_COLS = ["part1_country", "Locality_first", "EducationLevel", "Work", "Health_Prob",
                     "StressduringCovid", "FamilyProblems", "FinancialProblem", "SocialProblem",
                     "PsychologicalProb", "WorkStress", "FamilyIncome1", "part1_current_preg_first"]

In [2]:
dep_dir = os.path.join(PROCESSED_DIR, "arab5_depression")
train_dep = pd.read_csv(os.path.join(dep_dir, "train.csv"))
cal_dep = pd.read_csv(os.path.join(dep_dir, "calibration.csv"))
test_dep = pd.read_csv(os.path.join(dep_dir, "test.csv"))

dep_out = run_base_learner_suite(
    train_dep, cal_dep, test_dep, label_col="y_depression",
    numeric_cols=NUMERIC_COLS, categorical_cols=CATEGORICAL_COLS,
    model_names=MODEL_NAMES, trial_budget=TRIAL_BUDGET, seed=SEED, n_bootstrap=N_BOOTSTRAP,
    models_dir=os.path.join(MODELS_ROOT, "arab5_depression"), tables_dir=TABLES_DIR,
    dataset_tag="arab5_depression", svm_subsample_n=SVM_FIT_ROWS,
)
dep_out["results_df"]

[arab5_depression] Train/Cal/Test: (902, 21)/(296, 21)/(301, 21) | unique training combos: 870 of 902
[arab5_depression] dt: best CV AUC = 0.6714
[arab5_depression] rf: best CV AUC = 0.6899
[arab5_depression] gb: best CV AUC = 0.6850


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

[arab5_depression] svm: best CV AUC = 0.6776
[arab5_depression] nb: best CV AUC = 0.6658
[arab5_depression] knn: best CV AUC = 0.6795
[arab5_depression] xgb: best CV AUC = 0.6807


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

[arab5_depression] Done. Results saved to C:\maternal-mental-health-risk\reports/tables, models saved to C:\maternal-mental-health-risk\models\arab5_depression.


,model,threshold,accuracy,macro_f1,auc_roc,auc_ci_lower,auc_ci_upper,cv_auc_tuning
4,nb,0.89,0.6312,0.6270,0.6925,0.6266,0.7506,0.6658
1,rf,0.55,0.6379,0.6314,0.6867,0.6252,0.7468,0.6899
6,xgb,0.44,0.6113,0.6084,0.6745,0.6131,0.7367,0.6807
3,svm,0.56,0.6113,0.6020,0.6684,0.6061,0.7289,0.6776
2,gb,0.49,0.6113,0.6112,0.6602,0.5951,0.7220,0.6850
5,knn,0.47,0.5980,0.5974,0.6337,0.5680,0.6975,0.6795
0,dt,0.47,0.5980,0.5976,0.6319,0.5688,0.6936,0.6714


In [3]:
anx_dir = os.path.join(PROCESSED_DIR, "arab5_anxiety")
train_anx = pd.read_csv(os.path.join(anx_dir, "train.csv"))
cal_anx = pd.read_csv(os.path.join(anx_dir, "calibration.csv"))
test_anx = pd.read_csv(os.path.join(anx_dir, "test.csv"))

anx_out = run_base_learner_suite(
    train_anx, cal_anx, test_anx, label_col="y_anxiety",
    numeric_cols=NUMERIC_COLS, categorical_cols=CATEGORICAL_COLS,
    model_names=MODEL_NAMES, trial_budget=TRIAL_BUDGET, seed=SEED, n_bootstrap=N_BOOTSTRAP,
    models_dir=os.path.join(MODELS_ROOT, "arab5_anxiety"), tables_dir=TABLES_DIR,
    dataset_tag="arab5_anxiety", svm_subsample_n=SVM_FIT_ROWS,
)
anx_out["results_df"]

[arab5_anxiety] Train/Cal/Test: (565, 21)/(190, 21)/(191, 21) | unique training combos: 558 of 565
[arab5_anxiety] dt: best CV AUC = 0.7442
[arab5_anxiety] rf: best CV AUC = 0.7747
[arab5_anxiety] gb: best CV AUC = 0.7671


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

[arab5_anxiety] svm: best CV AUC = 0.7619
[arab5_anxiety] nb: best CV AUC = 0.7597
[arab5_anxiety] knn: best CV AUC = 0.7663
[arab5_anxiety] xgb: best CV AUC = 0.7652


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

[arab5_anxiety] Done. Results saved to C:\maternal-mental-health-risk\reports/tables, models saved to C:\maternal-mental-health-risk\models\arab5_anxiety.


,model,threshold,accuracy,macro_f1,auc_roc,auc_ci_lower,auc_ci_upper,cv_auc_tuning
5,knn,0.43,0.8010,0.7392,0.7976,0.7205,0.8699,0.7663
6,xgb,0.45,0.7853,0.7167,0.7848,0.7021,0.8626,0.7652
3,svm,0.42,0.7906,0.7418,0.7834,0.7021,0.8594,0.7619
1,rf,0.45,0.7958,0.7228,0.7815,0.6958,0.8631,0.7747
2,gb,0.42,0.7592,0.6490,0.7711,0.6868,0.8509,0.7671
4,nb,0.06,0.7435,0.6952,0.7625,0.6736,0.8444,0.7597
0,dt,0.45,0.7801,0.7155,0.7236,0.6343,0.8056,0.7442


In [4]:
fullN_dir = os.path.join(PROCESSED_DIR, "arab5_depression_fullN_sensitivity")
train_full = pd.read_csv(os.path.join(fullN_dir, "train.csv"))
cal_full = pd.read_csv(os.path.join(fullN_dir, "calibration.csv"))
test_full = pd.read_csv(os.path.join(fullN_dir, "test.csv"))

fullN_out = run_base_learner_suite(
    train_full, cal_full, test_full, label_col="y_depression",
    numeric_cols=NUMERIC_COLS, categorical_cols=CATEGORICAL_COLS,
    model_names=MODEL_NAMES, trial_budget=TRIAL_BUDGET, seed=SEED, n_bootstrap=N_BOOTSTRAP,
    models_dir=os.path.join(MODELS_ROOT, "arab5_depression_fullN_sensitivity"), tables_dir=TABLES_DIR,
    dataset_tag="arab5_depression_fullN_sensitivity", svm_subsample_n=SVM_FIT_ROWS,
)
fullN_out["results_df"]

[arab5_depression_fullN_sensitivity] Train/Cal/Test: (2132, 21)/(719, 21)/(718, 21) | unique training combos: 1837 of 2132
[arab5_depression_fullN_sensitivity] dt: best CV AUC = 0.8817
[arab5_depression_fullN_sensitivity] rf: best CV AUC = 0.9005
[arab5_depression_fullN_sensitivity] gb: best CV AUC = 0.8893


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

[arab5_depression_fullN_sensitivity] svm: best CV AUC = 0.8908
[arab5_depression_fullN_sensitivity] nb: best CV AUC = 0.8856
[arab5_depression_fullN_sensitivity] knn: best CV AUC = 0.8923
[arab5_depression_fullN_sensitivity] xgb: best CV AUC = 0.8963


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

[arab5_depression_fullN_sensitivity] Done. Results saved to C:\maternal-mental-health-risk\reports/tables, models saved to C:\maternal-mental-health-risk\models\arab5_depression_fullN_sensitivity.


,model,threshold,accuracy,macro_f1,auc_roc,auc_ci_lower,auc_ci_upper,cv_auc_tuning
1,rf,0.64,0.8134,0.7525,0.8920,0.8686,0.9138,0.9005
2,gb,0.13,0.7799,0.7473,0.8888,0.8652,0.9101,0.8893
5,knn,0.49,0.8092,0.7651,0.8879,0.8644,0.9103,0.8923
6,xgb,0.61,0.8231,0.7486,0.8877,0.8643,0.9092,0.8963
3,svm,0.80,0.8106,0.7470,0.8757,0.8509,0.9000,0.8908
0,dt,0.40,0.8064,0.7492,0.8729,0.8467,0.8969,0.8817
4,nb,0.30,0.8078,0.7616,0.8715,0.8461,0.8973,0.8856


In [5]:
for tag, out, out_dir in [("depression", dep_out, dep_dir),
                            ("anxiety", anx_out, anx_dir),
                            ("depression_fullN", fullN_out, fullN_dir)]:
    out["oof_predictions"].to_csv(os.path.join(out_dir, "oof_predictions_train.csv"), index=False)
    out["cal_predictions"].to_csv(os.path.join(out_dir, "base_predictions_calibration.csv"), index=False)
    out["test_predictions"].to_csv(os.path.join(out_dir, "base_predictions_test.csv"), index=False)
    print(f"Saved predictions for {tag}")

Saved predictions for depression
Saved predictions for anxiety
Saved predictions for depression_fullN


In [6]:
corr_dep = dep_out["oof_predictions"][MODEL_NAMES].corr()
print("Depression model correlations:")
print(corr_dep.round(3))

Depression model correlations:
        dt     rf     gb    svm     nb    knn    xgb
dt   1.000  0.877  0.890  0.798  0.775  0.697  0.871
rf   0.877  1.000  0.972  0.911  0.871  0.831  0.974
gb   0.890  0.972  1.000  0.889  0.856  0.778  0.954
svm  0.798  0.911  0.889  1.000  0.835  0.845  0.915
nb   0.775  0.871  0.856  0.835  1.000  0.756  0.867
knn  0.697  0.831  0.778  0.845  0.756  1.000  0.843
xgb  0.871  0.974  0.954  0.915  0.867  0.843  1.000


In [7]:
comparison = pd.merge(
    dep_out["results_df"][["model", "auc_roc", "auc_ci_lower", "auc_ci_upper"]],
    fullN_out["results_df"][["model", "auc_roc", "auc_ci_lower", "auc_ci_upper"]],
    on="model", suffixes=("_primary", "_fullN_sensitivity")
)
print(comparison)
comparison.to_csv(os.path.join(TABLES_DIR, "arab5_primary_vs_fullN_comparison.csv"), index=False)

  model  auc_roc_primary  auc_ci_lower_primary  auc_ci_upper_primary  \
0    nb           0.6925                0.6266                0.7506   
1    rf           0.6867                0.6252                0.7468   
2   xgb           0.6745                0.6131                0.7367   
3   svm           0.6684                0.6061                0.7289   
4    gb           0.6602                0.5951                0.7220   
5   knn           0.6337                0.5680                0.6975   
6    dt           0.6319                0.5688                0.6936   

   auc_roc_fullN_sensitivity  auc_ci_lower_fullN_sensitivity  \
0                     0.8715                          0.8461   
1                     0.8920                          0.8686   
2                     0.8877                          0.8643   
3                     0.8757                          0.8509   
4                     0.8888                          0.8652   
5                     0.8879           

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from preprocessing import build_preprocessor

ar_full = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "interim", "arab5_interim.csv"))
X_full = ar_full[NUMERIC_COLS + CATEGORICAL_COLS]
y_missingness = ar_full["dep_items_complete"].astype(int)

pre_check = build_preprocessor(NUMERIC_COLS, CATEGORICAL_COLS, scale_numeric=False)
rf_check = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=SEED, n_jobs=-1)
pipe_check = Pipeline([("pre", pre_check), ("clf", rf_check)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(pipe_check, X_full, y_missingness, cv=cv, scoring="roc_auc")
print(f"AUC predicting item-completion (i.e. real-vs-default-imputed label) from SDoH: {scores.mean():.4f}")

pipe_check.fit(X_full, y_missingness)
feat_names = pipe_check.named_steps["pre"].get_feature_names_out()
importances = pipe_check.named_steps["clf"].feature_importances_
top_idx = np.argsort(importances)[::-1][:8]
missingness_importance = pd.DataFrame({
    "feature": [feat_names[i] for i in top_idx],
    "importance": [round(importances[i], 4) for i in top_idx],
})
print(missingness_importance)

missingness_importance.to_csv(os.path.join(TABLES_DIR, "arab5_missingness_prediction_check.csv"), index=False)
print("\nCONCLUSION: item-missingness is highly predictable (AUC ~0.96) from SDoH features,")
print("driven primarily by StressduringCovid -- consistent with these variables belonging to")
print("the same survey module. This confirms the full-N sensitivity model's elevated AUC")
print("reflects module-completion signal, not genuine depression prediction. The primary,")
print("complete-items-only analysis (Cell 2) is the trustworthy result for this dataset.")

NameError: name 'Pipeline' is not defined

In [9]:
from sklearn.pipeline import Pipeline

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from preprocessing import build_preprocessor

ar_full = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "interim", "arab5_interim.csv"))
X_full = ar_full[NUMERIC_COLS + CATEGORICAL_COLS]
y_missingness = ar_full["dep_items_complete"].astype(int)

pre_check = build_preprocessor(NUMERIC_COLS, CATEGORICAL_COLS, scale_numeric=False)
rf_check = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=SEED, n_jobs=-1)
pipe_check = Pipeline([("pre", pre_check), ("clf", rf_check)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(pipe_check, X_full, y_missingness, cv=cv, scoring="roc_auc")
print(f"AUC predicting item-completion (i.e. real-vs-default-imputed label) from SDoH: {scores.mean():.4f}")

pipe_check.fit(X_full, y_missingness)
feat_names = pipe_check.named_steps["pre"].get_feature_names_out()
importances = pipe_check.named_steps["clf"].feature_importances_
top_idx = np.argsort(importances)[::-1][:8]
missingness_importance = pd.DataFrame({
    "feature": [feat_names[i] for i in top_idx],
    "importance": [round(importances[i], 4) for i in top_idx],
})
print(missingness_importance)

missingness_importance.to_csv(os.path.join(TABLES_DIR, "arab5_missingness_prediction_check.csv"), index=False)
print("\nCONCLUSION: item-missingness is highly predictable (AUC ~0.96) from SDoH features,")
print("driven primarily by StressduringCovid -- consistent with these variables belonging to")
print("the same survey module. This confirms the full-N sensitivity model's elevated AUC")
print("reflects module-completion signal, not genuine depression prediction. The primary,")
print("complete-items-only analysis (Cell 2) is the trustworthy result for this dataset.")

AUC predicting item-completion (i.e. real-vs-default-imputed label) from SDoH: 0.9611
                    feature  importance
0  cat__StressduringCovid_1      0.3226
1  cat__StressduringCovid_0      0.2893
2        num__SleepingAfter      0.0779
3   cat__FinancialProblem_1      0.0437
4   cat__FinancialProblem_0      0.0367
5  cat__PsychologicalProb_0      0.0323
6  cat__PsychologicalProb_1      0.0309
7         cat__WorkStress_0      0.0161

CONCLUSION: item-missingness is highly predictable (AUC ~0.96) from SDoH features,
driven primarily by StressduringCovid -- consistent with these variables belonging to
the same survey module. This confirms the full-N sensitivity model's elevated AUC
reflects module-completion signal, not genuine depression prediction. The primary,
complete-items-only analysis (Cell 2) is the trustworthy result for this dataset.
